# Second Hand Store: website-style dashboard
The results of the project shown as a web page: KPI cards, a forecast chart for every product, a results table and the answers to the 4 business questions.

**Note:** run the main notebook first. It creates the data in PostgreSQL. This notebook reads from the database, re-runs the forecast, builds the page with Python and shows it inside the notebook. It also saves the page as `dashboard.html`, so I can open it in the browser. The forecast cell takes about a minute.

## 1. Connect to the database

In [1]:
import html
import numpy as np
import pandas as pd
from IPython.display import display
from sqlalchemy import create_engine, text
from sklearn.ensemble import GradientBoostingRegressor

DB_URL = "postgresql+psycopg2://lauralukasevic@localhost:5432/project3_sql"
engine = create_engine(DB_URL)

def q(sql):
    return pd.read_sql(text(sql), engine)

print("Connected to:", q("SELECT current_database() AS db").iloc[0, 0])

Connected to: project3_sql


## 2. Load the sales history
**Note:** monthly units and revenue per product, straight from SQL. The unfinished month (Nov 2026) is left out.

In [2]:
products = q("SELECT product_id, name, category FROM products ORDER BY product_id")

monthly = q("""SELECT oi.product_id, date_trunc('month', o.order_date)::date AS month,
       SUM(oi.quantity)::int AS units, SUM(oi.quantity * oi.unit_price)::float AS revenue
FROM order_items oi JOIN orders o USING (order_id) GROUP BY 1, 2 ORDER BY 1, 2""")
monthly["month"] = pd.to_datetime(monthly["month"])
monthly = monthly[monthly.month < "2026-11-01"]

print(f"{len(monthly)} product-months loaded")

1415 product-months loaded


## 3. Forecasting functions
**Note:** the same machine learning code as in the main notebook (gradient boosting, one model per product). The explanation of how it works is there.

In [3]:
def features(s):
    d = pd.DataFrame({"y": s.values}, index=s.index)
    for l in (1, 2, 3, 6, 12):                          # Note: lag12 = same month last year
        d[f"lag{l}"] = d.y.shift(l)
    d["roll3"] = d.y.shift(1).rolling(3).mean()
    d["sin"], d["cos"] = np.sin(2 * np.pi * d.index.month / 12), np.cos(2 * np.pi * d.index.month / 12)
    d["t"] = np.arange(len(d))
    return d

def fit(s):
    d = features(s).dropna()
    model = GradientBoostingRegressor(n_estimators=300, max_depth=3, learning_rate=.05, subsample=.8, random_state=0)
    return model.fit(d.drop(columns="y"), d.y)

def forecast(s, horizon=12):
    model, s = fit(s), s.copy()
    for _ in range(horizon):                            # Note: predict one month, add it to history, predict the next
        nxt = s.index[-1] + pd.offsets.MonthBegin(1)
        s.loc[nxt] = np.nan
        x = features(s).iloc[[-1]].drop(columns="y")
        s.loc[nxt] = max(0, model.predict(x)[0])
    return s.iloc[-horizon:]

## 4. Run the forecasts
**Note:** for every product I keep the real sales, the backtest (what the model predicted for the last 12 months without seeing them), the forecast for the next 12 months and the error %.

In [4]:
results = []
for p in products.itertuples():
    m = monthly[monthly.product_id == p.product_id]
    s = m.set_index("month").units.asfreq("MS", fill_value=0).astype(float)
    bt = forecast(s.iloc[:-12])                         # Note: backtest = trained until Oct 2025, predicts the last 12 months
    actual = s.iloc[-12:]
    mape = float((abs(bt.values - actual.values) / actual.clip(lower=1).values).mean() * 100)
    fc = forecast(s)                                    # Note: the real forecast, Nov 2026 - Oct 2027
    price = m.tail(12).revenue.sum() / m.tail(12).units.sum()
    results.append({"name": p.name, "cat": p.category, "price": round(price, 2), "mape": round(mape, 1),
                    "hist": s.round().astype(int).tolist(),
                    "bt": bt.round().astype(int).tolist(),
                    "fc": fc.round().astype(int).tolist()})

print(f"Done: {len(results)} products forecast")

Done: 12 products forecast


## 5. Numbers for the page
**Note:** totals from SQL (orders, items, customers, revenue) and revenue per year and category.

In [5]:
kpi = q("""SELECT (SELECT COUNT(*) FROM orders) AS orders,
                  (SELECT SUM(quantity) FROM order_items) AS items,
                  (SELECT COUNT(*) FROM customers) AS customers,
                  (SELECT ROUND(SUM(quantity * unit_price)) FROM order_items)::float AS revenue""").iloc[0]

rev = q("""SELECT EXTRACT(YEAR FROM o.order_date)::int AS year, p.category,
       SUM(oi.quantity * oi.unit_price)::float AS revenue
FROM order_items oi JOIN orders o USING (order_id) JOIN products p USING (product_id)
GROUP BY 1, 2""").pivot(index="year", columns="category", values="revenue").round().astype(int)

years, cats, rev_rows = rev.index.tolist(), rev.columns.tolist(), rev.values.tolist()
rev_2025 = monthly[monthly.month.dt.year == 2025].revenue.sum()

kpi

orders         41134.0
items          41563.0
customers       2500.0
revenue      1891004.0
Name: 0, dtype: float64

## 6. Chart functions
**Note:** every chart is drawn by a Python function that returns SVG (a picture made of text). I don't need to change these.

In [6]:
FULL_MONTHS = ["January", "February", "March", "April", "May", "June", "July", "August", "September", "October", "November", "December"]
MONTHS = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
COLORS = ["#2d5b8a", "#c9822b", "#7a9e7e", "#b5617a", "#8a7bb5", "#5aa9a4"]   # category colors

def money(n):
    return f"{n:,.0f}"

def svg_forecast(r):
    """Line chart for one product: actual (solid), backtest (dotted), forecast (dashed)."""
    H = r["hist"][-60:]                                  # Note: last 5 years of real sales
    n = len(H); N = n + 12
    W, Ht, L, B, T = 900, 300, 40, 26, 10                # Note: chart size and margins
    mx = max(H + r["fc"] + r["bt"]) * 1.1
    start = len(r["hist"]) - n
    x = lambda k: L + k * (W - L - 8) / (N - 1)
    y = lambda v: T + (Ht - T - B) * (1 - v / mx)
    def line(arr, off):
        return "".join(("L" if k else "M") + f"{x(k + off):.1f},{y(v):.1f}" for k, v in enumerate(arr))
    s = ""
    for g in range(5):                                   # Note: grid lines and the numbers on the left
        v = mx * g / 4
        s += (f'<line x1="{L}" x2="{W}" y1="{y(v):.1f}" y2="{y(v):.1f}" stroke="var(--line)"/>'
              f'<text x="{L - 6}" y="{y(v) + 4:.1f}" text-anchor="end">{int(v + .5)}</text>')
    for k in range(N):                                   # Note: a label at every January
        m = (start + k) % 12
        if m == 0 or (k == 0 and (12 - m) % 12 > 3):    # Note: skip the first label if it would touch the next January
            s += f'<text x="{x(k):.1f}" y="{Ht - 6}" text-anchor="middle">{MONTHS[m]} {2017 + (start + k) // 12}</text>'
    s += f'<path d="{line(H, 0)}" fill="none" stroke="var(--a)" stroke-width="2.2"/>'
    s += f'<path d="{line(r["bt"], n - 12)}" fill="none" stroke="var(--ink)" stroke-opacity=".5" stroke-width="1.8" stroke-dasharray="2 3"/>'
    s += f'<path d="{line([H[-1]] + r["fc"], n - 1)}" fill="none" stroke="var(--f)" stroke-width="2.6" stroke-dasharray="7 4"/>'
    return f'<svg viewBox="0 0 {W} {Ht}" role="img" aria-label="Monthly sales and forecast">{s}</svg>'

def svg_hbar(items, fmt, color, label):
    """Horizontal bars. items = [(name, value), ...]"""
    n, lw, W = len(items), 170, 900
    vs = [v for _, v in items]
    mn, mx = min(0, *vs), max(0, *vs)
    pad = 50 if mn < 0 else 0                            # Note: extra room on the left for negative bars
    sc = (W - lw - 80 - pad) / ((mx - mn) or 1)
    z = lw + pad + (-mn) * sc                                  # Note: where the zero line is
    s = ""
    for i, (name, v) in enumerate(items):
        y = i * 26 + 4
        w = abs(v) * sc
        x0 = z - w if v < 0 else z
        tx = x0 - 5 if v < 0 else x0 + w + 5
        s += (f'<text x="{lw - 8}" y="{y + 14}" text-anchor="end">{name}</text>'
              f'<rect x="{x0:.1f}" y="{y}" width="{w:.1f}" height="18" rx="3" fill="{color(v)}"/>'
              f'<text x="{tx:.1f}" y="{y + 14}" text-anchor="{"end" if v < 0 else "start"}">{fmt(v)}</text>')
    s += f'<line x1="{z:.1f}" x2="{z:.1f}" y1="0" y2="{n * 26 + 4}" stroke="var(--mute)"/>'
    return f'<svg viewBox="0 0 {W} {n * 26 + 8}" role="img" aria-label="{label}">{s}</svg>'

def svg_heatmap(results):
    """Busy months: every row is compared with that product's own average."""
    MN = [MONTHS[(10 + i) % 12] for i in range(12)]      # Note: forecast starts in November
    rows = [(r["name"], r["fc"]) for r in results]
    rows.append(("All products", [sum(r["fc"][i] for r in results) for i in range(12)]))
    hl = 170; cw = (900 - hl) / 12
    s = "".join(f'<text x="{hl + i * cw + cw / 2:.1f}" y="14" text-anchor="middle">{m}</text>' for i, m in enumerate(MN))
    for j, (name, vals) in enumerate(rows):
        mean, mxv = sum(vals) / 12, max(vals)
        y = 24 + j * 24 + (8 if j == len(results) else 0)
        s += f'<text x="{hl - 8}" y="{y + 15}" text-anchor="end">{name}</text>'
        for i, v in enumerate(vals):
            ix = v / mean
            opacity = min(1, max(.06, ix - .5))          # Note: darker = busier
            peak = ' stroke="var(--f)" stroke-width="2"' if v == mxv else ""
            s += (f'<rect x="{hl + i * cw + 1:.1f}" y="{y}" width="{cw - 2:.1f}" height="22" rx="3" '
                  f'fill="var(--a)" fill-opacity="{opacity:.2f}"{peak}>'
                  f'<title>{name}, {MN[i]}: {v} ({ix:.2f}x average)</title></rect>')
    return f'<svg viewBox="0 0 900 {24 + len(rows) * 24 + 14}" role="img" aria-label="Seasonality heatmap">{s}</svg>'

def svg_revenue(years, cats, rev_rows):
    """Stacked bars: revenue per year, split by category."""
    tmax = max(sum(row) for row in rev_rows) * 1.08
    bw = (900 - 50) / len(years)
    s = ""
    for i, row in enumerate(rev_rows):
        acc = 0
        for j, v in enumerate(row):
            h = v / tmax * 240
            yy = 250 - (acc + v) / tmax * 240
            s += (f'<rect x="{50 + i * bw + 8:.1f}" y="{yy:.1f}" width="{bw - 16:.1f}" height="{h:.1f}" fill="{COLORS[j]}">'
                  f'<title>{years[i]} {cats[j]}: €{money(v)}</title></rect>')
            acc += v
        total = sum(row)
        cx = 50 + i * bw + bw / 2
        s += (f'<text x="{cx:.1f}" y="270" text-anchor="middle">{years[i]}</text>'
              f'<text x="{cx:.1f}" y="{250 - total / tmax * 240 - 5:.1f}" text-anchor="middle">€{int(total / 1000 + .5)}k</text>')
    legend = "".join(f'<span><i style="background:{COLORS[j]}"></i>{c}</span>' for j, c in enumerate(cats))
    return (f'<svg viewBox="0 0 900 280" role="img" aria-label="Yearly revenue by category">{s}</svg>'
            f'<div class="legend">{legend}</div>')

## 7. Page design
**Note:** the colors, fonts and layout of the page (CSS), plus a tiny script so the product buttons switch the chart. I don't need to change this.

In [7]:
CSS = """
:root{--bg:#eef1f4;--card:#fff;--ink:#1b2733;--mute:#5d6b79;--line:#d5dce3;--a:#2d5b8a;--f:#c9822b;--tag:#f2c94c;box-sizing:border-box;padding-top:env(safe-area-inset-top,0px);padding-bottom:env(safe-area-inset-bottom,0px)}
@media(prefers-color-scheme:dark){:root:not([data-theme="light"]){--bg:#121a22;--card:#1a2530;--ink:#e8edf2;--mute:#9aa8b6;--line:#2e3d4b;--a:#6aa3da;--f:#e8a049;--tag:#d9b43f}}
*{box-sizing:border-box}html{scroll-padding-top:env(safe-area-inset-top,0px)}
body{margin:0;background:var(--bg);color:var(--ink);font:16px/1.55 "Bricolage Grotesque",system-ui,sans-serif}
main{max-width:1040px;margin:0 auto;padding:24px 18px 60px}
h1{font-size:clamp(2.2rem,7vw,4rem);line-height:1;margin:0 0 12px;font-weight:800;letter-spacing:-.02em}
h2{font-size:1.35rem;margin:42px 0 6px}p{margin:0;color:var(--mute);max-width:62ch}
.hero{display:flex;gap:28px;align-items:center;flex-wrap:wrap;margin-top:20px}
.tag{position:relative;background:var(--tag);color:#1b2733;padding:22px 26px 22px 46px;border-radius:6px 28px 28px 6px;transform:rotate(-3deg);box-shadow:0 2px 0 rgba(0,0,0,.15)}
.tag:before{content:"";position:absolute;left:16px;top:50%;width:14px;height:14px;margin-top:-7px;border-radius:50%;background:var(--bg)}
.tag b{display:block;font-size:2.6rem;line-height:1}.tag span{font-size:.9rem}
.kpis{display:grid;grid-template-columns:repeat(auto-fit,minmax(150px,1fr));gap:12px;margin-top:28px}
.k{background:var(--card);border:1px solid var(--line);border-radius:10px;padding:14px}.k b{display:block;font-size:1.6rem}.k span{color:var(--mute);font-size:.85rem}
.panel{background:var(--card);border:1px solid var(--line);border-radius:10px;padding:16px;margin-top:14px}
.chips{display:flex;flex-wrap:wrap;gap:8px;margin-bottom:12px}
.chips button{font:inherit;font-size:.85rem;border:1px solid var(--line);background:none;color:var(--ink);padding:5px 12px;border-radius:999px;cursor:pointer}
.chips button[aria-pressed=true]{background:var(--a);border-color:var(--a);color:#fff}
.chips button:focus-visible{outline:3px solid var(--f);outline-offset:2px}
svg{width:100%;height:auto;display:block}svg text{fill:var(--mute);font:11px "Bricolage Grotesque",sans-serif}
.legend{display:flex;gap:16px;flex-wrap:wrap;font-size:.85rem;color:var(--mute);margin-top:8px}.legend i{display:inline-block;width:18px;height:3px;margin-right:6px;vertical-align:middle}
.stats{display:flex;gap:24px;flex-wrap:wrap;margin:6px 0 10px}.stats b{font-size:1.4rem;display:block}.stats span{font-size:.85rem;color:var(--mute)}
.scroll{overflow-x:auto}table{border-collapse:collapse;width:100%;font-size:.92rem}th,td{padding:8px 10px;border-bottom:1px solid var(--line);text-align:right;white-space:nowrap}th:first-child,td:first-child{text-align:left}th{color:var(--mute);font-weight:600}
tr[data-i]{cursor:pointer}tr[data-i]:hover{background:var(--bg)}.up{color:#2e8b57}.dn{color:#c0504d}
.q h3{font-size:1.05rem;margin:0 0 4px}.q p{margin-bottom:10px}.q .do{color:var(--ink);font-weight:600}.note{font-size:.85rem;margin-top:6px}
footer{margin-top:40px;font-size:.85rem;color:var(--mute)}
"""

# Note: a tiny script so the product buttons and the table rows switch the chart
SCRIPT = """
function fit(){var f=window.frameElement;if(f){f.style.height=document.documentElement.scrollHeight+'px'}}
function sel(i){
  document.querySelectorAll('.pane').forEach(function(e){e.hidden=(e.dataset.i!=i)});
  document.querySelectorAll('#chips button').forEach(function(b){b.setAttribute('aria-pressed',b.dataset.i==i)});
  fit();
}
document.getElementById('chips').onclick=function(e){if(e.target.dataset.i!=null)sel(+e.target.dataset.i)};
document.getElementById('tbl').onclick=function(e){
  var r=e.target.closest('tr[data-i]');
  if(r){sel(+r.dataset.i);document.getElementById('fc').scrollIntoView({behavior:'smooth',block:'center'})}
};
window.addEventListener('load',fit);window.addEventListener('resize',fit);
"""

## 8. Build the page
**Note:** the answers to the 4 business questions are written from the numbers, so the text always matches the charts.

In [8]:
# Note: small helpers for the numbers of every product
last12 = lambda r: sum(r["hist"][-12:])
nxt12 = lambda r: sum(r["fc"])
chg = lambda r: (nxt12(r) / last12(r) - 1) * 100             # forecast change vs last 12 months, %
bias = lambda r: (sum(r["bt"]) / last12(r) - 1) * 100        # backtest bias, % (negative = model too low)
frev = lambda r: round(nxt12(r) * r["price"])                # forecast revenue, €
pct = lambda v: f"{'+' if v > 0 else ''}{v:.1f}%"
ranked = lambda f: sorted([(r["name"], f(r)) for r in results], key=lambda t: t[1], reverse=True)

total_items = sum(nxt12(r) for r in results)
total_rev = sum(frev(r) for r in results)

# Note: the answers to the 4 questions are written from the numbers, so they always match the charts
top3 = sorted(results, key=frev, reverse=True)[:3]
q1 = (f"{top3[0]['name']}, {top3[1]['name']} and {top3[2]['name']} make about €{round(sum(frev(r) for r in top3) / 3 / 1000)}k each. "
      f"The whole store is forecast at about €{round(total_rev / 1000)}k, {'up' if total_rev >= rev_2025 else 'down'} "
      f"from €{round(rev_2025 / 1000)}k in 2025. ")

grow = sorted(results, key=chg, reverse=True)[:3]
decline = [r for r in results if chg(r) < -5]
q2 = (f"{grow[0]['name']} ({chg(grow[0]):+.0f}%), {grow[1]['name']} ({chg(grow[1]):+.0f}%) and {grow[2]['name']} ({chg(grow[2]):+.0f}%) grow the most. ")
if len(decline) == 1:
    q2 += f"The {decline[0]['name']} is the only clear decline (−{abs(chg(decline[0])):.0f}%). "
elif decline:
    q2 += "Clear declines: " + ", ".join(f"{r['name']} (−{abs(chg(r)):.0f}%)" for r in decline) + ". "

by_month = [sum(r["fc"][i] for r in results) for i in range(12)]     # Note: Nov, Dec, ... Oct
cal = lambda i: (10 + i) % 12                                         # position in the list -> calendar month
top2 = sorted(sorted(range(12), key=lambda i: by_month[i], reverse=True)[:2], key=cal)
quiet = min(range(12), key=lambda i: by_month[i])
q3 = f"Store-wide sales peak in {FULL_MONTHS[cal(top2[0])]} and {FULL_MONTHS[cal(top2[1])]} and are quietest in {FULL_MONTHS[cal(quiet)]}, but each product has its own season. "

clothing = [r["mape"] for r in results if r["cat"] == "Clothing"]
high = [r for r in results if r["mape"] > 40]
per_month = [last12(r) / 12 for r in high]
too_low = sum(1 for r in results if bias(r) < 0)
q4 = (f"Clothing is the most reliable (error {min(clothing):.0f}–{max(clothing):.0f}%). "
      f"{' and '.join(r['name'] for r in high)} are above 40% because they sell only {min(per_month):.0f}–{max(per_month):.0f} a month. "
      f"In the backtest the model was too low for {too_low} of {len(results)} products, so ")

# ---- the pieces of the page ----
kpis_html = "".join(f'<div class="k"><b>{v}</b><span>{label}</span></div>' for label, v in [
    ("Items sold", money(kpi["items"])), ("Orders", money(kpi["orders"])),
    ("Customers", money(kpi["customers"])), ("Revenue", "€" + money(kpi["revenue"]))])

chips = "".join(f'<button data-i="{i}" aria-pressed="{"true" if i == 0 else "false"}">{r["name"]}</button>'
                for i, r in enumerate(results))

panes = ""
for i, r in enumerate(results):
    a, f = last12(r), nxt12(r)
    stats = (f'<div class="stats"><div><b>{money(a)}</b><span>sold, last 12 months</span></div>'
             f'<div><b>{money(f)}</b><span>forecast, next 12 months</span></div>'
             f'<div><b>€{money(round(f * r["price"]))}</b><span>forecast revenue (avg €{r["price"]})</span></div></div>')
    panes += f'<div class="pane" data-i="{i}"{"" if i == 0 else " hidden"}>{stats}<div class="scroll">{svg_forecast(r)}</div></div>'

table = ("<tr><th>Product</th><th>Category</th><th>Last 12 mo</th><th>Next 12 mo</th><th>Change</th><th>Forecast error</th></tr>"
         + "".join(f'<tr data-i="{i}"><td>{r["name"]}</td><td>{r["cat"]}</td><td>{money(last12(r))}</td><td>{money(nxt12(r))}</td>'
                   f'<td class="{"up" if chg(r) >= 0 else "dn"}">{"+" if chg(r) >= 0 else ""}{chg(r):.1f}%</td><td>{r["mape"]:g}%</td></tr>'
                   for i, r in enumerate(results)))

chart_q1 = svg_hbar(ranked(frev), lambda v: "€" + money(v), lambda v: "var(--a)", "Forecast revenue by product")
chart_q2 = svg_hbar(ranked(chg), pct, lambda v: "#2e8b57" if v >= 0 else "#c0504d", "Forecast change by product")
chart_q4a = svg_hbar(ranked(bias), pct, lambda v: "var(--a)" if v >= 0 else "var(--f)", "Backtest bias by product")
chart_q4b = svg_hbar(ranked(lambda r: r["mape"]), lambda v: f"{v:g}%", lambda v: "var(--f)" if v > 35 else "var(--a)", "Forecast error by product")

LEGEND = ('<div class="legend"><span><i style="background:var(--a)"></i>Actual</span>'
          '<span><i style="background:var(--ink);opacity:.5"></i>Backtest</span>'
          '<span><i style="background:var(--f)"></i>Forecast</span></div>')

body = f"""<main>
<div class="hero"><div><h1>Second Hand Store</h1><p>A second-hand store selling clothes, shoes, furniture, media, electronics and home goods. Below: ten years of sales (2017–2026) and a machine-learning forecast for the next 12 months, product by product.</p></div>
<div class="tag"><b>{total_items:,}</b><span>items forecast to sell,<br>Nov 2026 – Oct 2027</span></div></div>
<div class="kpis">{kpis_html}</div>

<h2>Sales forecast by product</h2>
<p>Pick a product. Solid line is actual monthly sales, dotted is what the model predicted for the last 12 months when it hadn’t seen them, dashed is the forecast ahead.</p>
<div class="panel"><div class="chips" id="chips">{chips}</div><div id="fc">{panes}</div>
{LEGEND}</div>

<h2>All products, next 12 months</h2>
<p>Select a row to see its chart. Forecast error is the average % miss on the held-out last 12 months, so lower is better.</p>
<div class="panel scroll"><table id="tbl">{table}</table></div>

<h2>Revenue by category</h2>
<p>Yearly revenue in euros. 2020 dips with the pandemic; 2026 is through 14 November.</p>
<div class="panel scroll">{svg_revenue(years, cats, rev_rows)}</div>

<h2>Four business questions</h2>
<p>The answers come straight from the forecast above. Each one ends with what I would do about it.</p>
<div class="panel q"><h3>1. Which products will bring in the most revenue next year?</h3><p>{q1}<span class="do">Put the most sourcing effort and shelf space into these three.</span></p>{chart_q1}</div>
<div class="panel q"><h3>2. Which products are growing and which are shrinking?</h3><p>{q2}<span class="do">Source more books and bookshelves, and review the armchair’s price and sourcing.</span></p>{chart_q2}</div>
<div class="panel q"><h3>3. When should we stock up?</h3><p>{q3}<span class="do">Buy winter coats from September and sneakers from March.</span></p><div class="scroll">{svg_heatmap(results)}</div><p class="note">Darker means busier than that product’s own average. The amber outline marks its peak month.</p></div>
<div class="panel q"><h3>4. Which forecasts can we trust?</h3><p>{q4}<span class="do">plan stock from the clothing forecasts and order a bit more than the model says.</span></p>
<p class="note">Bias: negative means the model predicted less than what really sold.</p>{chart_q4a}
<p class="note" style="margin-top:14px">Forecast error: amber means above 35%, so treat it as a rough range.</p>{chart_q4b}</div>

<footer>Data is synthetic, generated in Python and stored in PostgreSQL. Model: gradient boosting on lagged sales and seasonality, one model per product.</footer>
</main>"""

PAGE = ('<!DOCTYPE html><html lang="en"><head><meta charset="utf-8">'
        '<meta name="viewport" content="width=device-width, initial-scale=1, viewport-fit=cover">'
        '<title>Second Hand Store – Sales forecast and business questions</title>'
        '<link href="https://fonts.googleapis.com/css2?family=Bricolage+Grotesque:wght@400;600;800&display=swap" rel="stylesheet">'
        f'<style>{CSS}</style></head><body>{body}<script>{SCRIPT}</script></body></html>')

with open("dashboard.html", "w", encoding="utf-8") as f:
    f.write(PAGE)
print(f"Page built: {len(PAGE):,} characters, saved as dashboard.html")

Page built: 84,728 characters, saved as dashboard.html


## 9. Show the page

In [9]:
# Note: the page is shown in a frame, so its own design (colors, fonts) stays the same as in the browser
frame = (f'<iframe srcdoc="{html.escape(PAGE, quote=True)}" title="Second Hand Store dashboard" '
         f'style="width:100%;height:1400px;border:0"></iframe>')
display({"text/html": frame}, raw=True)

<iframe srcdoc="<!DOCTYPE html><html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width, initial-scale=1, viewport-fit=cover"><title>Second Hand Store – Sales forecast and business questions</title><link href="https://fonts.googleapis.com/css2?family=Bricolage+Grotesque:wght@400;600;800&display=swap" rel="stylesheet"><style>
:root{--bg:#eef1f4;--card:#fff;--ink:#1b2733;--mute:#5d6b79;--line:#d5dce3;--a:#2d5b8a;--f:#c9822b;--tag:#f2c94c;box-sizing:border-box;padding-top:env(safe-area-inset-top,0px);padding-bottom:env(safe-area-inset-bottom,0px)}
@media(prefers-color-scheme:dark){:root:not([data-theme="light"]){--bg:#121a22;--card:#1a2530;--ink:#e8edf2;--mute:#9aa8b6;--line:#2e3d4b;--a:#6aa3da;--f:#e8a049;--tag:#d9b43f}}
*{box-sizing:border-box}html{scroll-padding-top:env(safe-area-inset-top,0px)}
body{margin:0;background:var(--bg);color:var(--ink);font:16px/1.55 "Bricolage Grotesque",system-ui,sans-serif}
main{max-width:1040px;margin:0 auto;padding:24px 18px 60px}
h1{font-size:clamp(2.2rem,7vw,4rem);line-height:1;margin:0 0 12px;font-weight:800;letter-spacing:-.02em}
h2{font-size:1.35rem;margin:42px 0 6px}p{margin:0;color:var(--mute);max-width:62ch}
.hero{display:flex;gap:28px;align-items:center;flex-wrap:wrap;margin-top:20px}
.tag{position:relative;background:var(--tag);color:#1b2733;padding:22px 26px 22px 46px;border-radius:6px 28px 28px 6px;transform:rotate(-3deg);box-shadow:0 2px 0 rgba(0,0,0,.15)}
.tag:before{content:"";position:absolute;left:16px;top:50%;width:14px;height:14px;margin-top:-7px;border-radius:50%;background:var(--bg)}
.tag b{display:block;font-size:2.6rem;line-height:1}.tag span{font-size:.9rem}
.kpis{display:grid;grid-template-columns:repeat(auto-fit,minmax(150px,1fr));gap:12px;margin-top:28px}
.k{background:var(--card);border:1px solid var(--line);border-radius:10px;padding:14px}.k b{display:block;font-size:1.6rem}.k span{color:var(--mute);font-size:.85rem}
.panel{background:var(--card);border:1px solid var(--line);border-radius:10px;padding:16px;margin-top:14px}
.chips{display:flex;flex-wrap:wrap;gap:8px;margin-bottom:12px}
.chips button{font:inherit;font-size:.85rem;border:1px solid var(--line);background:none;color:var(--ink);padding:5px 12px;border-radius:999px;cursor:pointer}
.chips button[aria-pressed=true]{background:var(--a);border-color:var(--a);color:#fff}
.chips button:focus-visible{outline:3px solid var(--f);outline-offset:2px}
svg{width:100%;height:auto;display:block}svg text{fill:var(--mute);font:11px "Bricolage Grotesque",sans-serif}
.legend{display:flex;gap:16px;flex-wrap:wrap;font-size:.85rem;color:var(--mute);margin-top:8px}.legend i{display:inline-block;width:18px;height:3px;margin-right:6px;vertical-align:middle}
.stats{display:flex;gap:24px;flex-wrap:wrap;margin:6px 0 10px}.stats b{font-size:1.4rem;display:block}.stats span{font-size:.85rem;color:var(--mute)}
.scroll{overflow-x:auto}table{border-collapse:collapse;width:100%;font-size:.92rem}th,td{padding:8px 10px;border-bottom:1px solid var(--line);text-align:right;white-space:nowrap}th:first-child,td:first-child{text-align:left}th{color:var(--mute);font-weight:600}
tr[data-i]{cursor:pointer}tr[data-i]:hover{background:var(--bg)}.up{color:#2e8b57}.dn{color:#c0504d}
.q h3{font-size:1.05rem;margin:0 0 4px}.q p{margin-bottom:10px}.q .do{color:var(--ink);font-weight:600}.note{font-size:.85rem;margin-top:6px}
footer{margin-top:40px;font-size:.85rem;color:var(--mute)}
</style></head><body><main>
<div class="hero"><div><h1>Second Hand Store</h1><p>A second-hand store selling clothes, shoes, furniture, media, electronics and home goods. Below: ten years of sales (2017–2026) and a machine-learning forecast for the next 12 months, product by product.</p></div>
<div class="tag"><b>5,526</b><span>items forecast to sell,<br>Nov 2026 – Oct 2027</span></div></div>
<div class="kpis"><div class="k"><b>41,563</b><span>Items sold</span></div><div class="k"><b>41,134</b><span>Orders</span></div><div class="k"><b>2,500<